# Laptop Price Classification

**Goal:** classify laptops into three price tiers (Cheap, Affordable, Expensive) from their specifications.

**Dataset:** about 1,300 laptops with company, type, screen, CPU, RAM, storage, GPU, operating system, weight and price (in Indian rupees).

**Source:** public laptop price dataset from [Kaggle](https://www.kaggle.com/).
<!-- TODO: replace with the exact dataset title, owner, link and license once you have found the Kaggle page. -->

**Workflow:** inspect the data, clean it, explore it, engineer features, train models, and interpret the results.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

In [ ]:
df = pd.read_csv('laptop_data.csv', index_col=0)
print(df.shape)
df.describe()

# 1. Inspect the data quality

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

# 2. Clean the columns

In [ ]:
df['Ram'] = df['Ram'].astype(str).str.replace('GB', '').astype(int)
df['Weight'] = df['Weight'].astype(str).str.replace('kg', '').astype(float)

The prices in the source data are in Indian rupees (INR). They are converted to USD with an approximate exchange rate of 0.0105 USD per INR to make the values easier to read. The data was probably collected years ago, so the USD values are approximate. This does not affect the modeling, because the price tiers are defined on the converted values.

In [ ]:
INR_TO_USD = 0.0105   # approximate exchange rate

if 'Price' in df.columns:
    df['Price_USD'] = (df['Price'] * INR_TO_USD).round(2)
    df = df.drop(columns=['Price'])

# 3. Exploratory Data Analysis

## 3.1 Categorical features

In [ ]:
df['Company'].value_counts()

Several companies have very few records (fewer than 20 laptops each), such as Samsung, Razer, Mediacom and so on. Their average prices would not be reliable, and the model could memorize them (overfitting). To prevent this, these companies are grouped into a single category called “Other”.

In [ ]:
counts = df['Company'].value_counts()
rare = counts[counts < 20].index.tolist()
print(rare)
df['Company'] = df['Company'].replace(rare, 'Other')
df['Company'].value_counts()

In [ ]:
df['TypeName'].value_counts()

All laptop types have more than 20 records, so no grouping is needed for TypeName. Notebook is the most common type, while Workstation and Netbook have the least data.

## 3.2 Price by category

In [ ]:
df.groupby('Company')['Price_USD'].mean().sort_values()

In [ ]:
df.groupby('Ram')['Price_USD'].mean().sort_values()

In [ ]:
df.groupby('TypeName')['Price_USD'].agg(['mean', 'median', 'count']).sort_values('mean', ascending=False).round(2)

For most laptop types the mean price is higher than the median, which means the price distribution is right-skewed: a few very expensive laptops pull the average up. Ultrabook is the exception, with a mean close to its median.

## 3.3 Correlation

In [ ]:
df.corr(numeric_only=True)

In [ ]:
df.corr(numeric_only=True)['Price_USD'].sort_values(ascending=False)

Inches and Weight have the strongest correlation (0.82), as larger laptops tend to be heavier. Ram has the strongest correlation with Price_USD (0.74), making it the most useful numeric feature for price prediction. The other numeric features show little linear correlation with Price_USD.

## 3.4 Price distribution and outliers

In [ ]:
price_fig = px.histogram(df, x='Price_USD', nbins=40, marginal='box',
                         title='Distribution of Laptop Price (USD)')
price_fig.update_layout(bargap=0.05, width=900, height=600, title_x=0.5)
price_fig.show()

In [ ]:
Q1 = df['Price_USD'].quantile(0.25)
Q3 = df['Price_USD'].quantile(0.75)
IQR = Q3 - Q1
upper = Q3 + 1.5 * IQR

outliers = df[df['Price_USD'] > upper]
print(f'{len(outliers)} outliers above {upper:.2f} USD')
outliers[['Company', 'TypeName', 'Ram', 'Price_USD']].sort_values('Price_USD', ascending=False)

<!-- The numbers in this cell come from the previous run; re-check them after re-running the notebook. -->
The price distribution is right-skewed. Using the IQR method, 29 laptops (about 2% of the data) have prices above 1578.3 USD. Most of them are Gaming laptops with 32-64 GB of RAM or Workstations, so their high prices are consistent with their specifications. These are genuine high-end laptops rather than data errors, so they are kept. All of them fall into the Expensive tier, so they do not distort the classification target.

In [ ]:
figs = [
    px.box(df, x='TypeName', y='Price_USD', title='Price by Laptop Type',
           color_discrete_sequence=px.colors.qualitative.Antique),
    px.box(df, x='Company', y='Price_USD', title='Price by Company',
           color_discrete_sequence=px.colors.qualitative.Dark2),
    px.box(df, x='Ram', y='Price_USD', title='Price by RAM',
           color_discrete_sequence=px.colors.qualitative.T10),
    px.imshow(df.corr(numeric_only=True), text_auto='.2f', title='Correlation Heatmap',
              color_continuous_scale='Blues'),
]

for fig in figs:
    fig.update_layout(width=900, height=500, title_x=0.5)
    fig.show()

The boxplots show how price varies with laptop type, company and RAM, and the heatmap confirms that RAM has the strongest linear relationship with price.

In [ ]:
def outliers_per_group(df, col):
    result = []
    for name, g in df.groupby(col):
        Q1, Q3 = g['Price_USD'].quantile([0.25, 0.75])
        IQR = Q3 - Q1
        out = g[(g['Price_USD'] > Q3 + 1.5 * IQR) | (g['Price_USD'] < Q1 - 1.5 * IQR)]
        result.append(out)
    return pd.concat(result)

out_type = outliers_per_group(df, 'TypeName')
print(out_type['TypeName'].value_counts())
out_comp = outliers_per_group(df, 'Company')
print(out_comp['Company'].value_counts())
out_ram = outliers_per_group(df, 'Ram')
print(out_ram['Ram'].value_counts())

In [ ]:
out_ram[out_ram['Ram'] == 4][['Company', 'TypeName', 'Inches', 'Price_USD']].sort_values('Price_USD', ascending=False)

In [ ]:
df.loc[[981, 1004]]

<!-- The numbers in this cell come from the previous run; re-check them after re-running the notebook. -->
Outliers were also checked within each group. Dell has the most company-level outliers (19, about 6% of its laptops), suggesting a wide price range from budget to high-end models. For RAM, the 4 GB and 8 GB groups have the most outliers (15 and 9), meaning some low-RAM laptops are priced well above their peers, likely due to other specifications such as the processor, storage or laptop type. Because these prices are consistent with the laptops' specifications, the outliers are kept in the dataset.

Two Toshiba laptops (rows 981 and 1004) have identical specifications and prices and differ only in operating system (Windows 7 and Windows 10). They are valid, separate records, not duplicates, so both are kept.

## 3.5 Operating system

In [ ]:
df.groupby('OpSys')['Price_USD'].agg(['mean', 'median', 'count']).round(2)

<!-- The numbers in this cell come from the previous run; re-check them after re-running the notebook. -->
Windows 7 laptops are priced noticeably higher than Windows 10 laptops (median about 836 vs 570 USD), so they are kept as a separate category. Windows 10 S (only 8 records) is merged into Windows 10, “Mac OS X” and “macOS” are unified, and Android (2 records) is grouped into “Other”.

In [ ]:
df['OpSys'] = df['OpSys'].replace({
    'Windows 10 S': 'Windows 10',
    'Mac OS X': 'macOS',
    'Android': 'Other',
})
df['OpSys'].value_counts()

# 4. Feature Engineering

## 4.1 Target: price tier

The target is a price tier with three classes: **Cheap** (below 500 USD), **Affordable** (500 to 800 USD) and **Expensive** (800 USD and above). The thresholds follow common budget, mid-range and premium segments. A separate “Premium” tier above 1,500 USD contained only 41 laptops, too few for a model to learn from, so it was merged into Expensive.

In [ ]:
conditions = [
    df['Price_USD'] >= 800,
    (df['Price_USD'] >= 500) & (df['Price_USD'] < 800),
    df['Price_USD'] < 500,
]
tier_labels = ['Expensive', 'Affordable', 'Cheap']
df['Price_tier'] = np.select(conditions, tier_labels, default='Unknown')

print((df['Price_tier'] == 'Unknown').sum(), 'laptops without a tier')
df['Price_tier'].value_counts()

In [ ]:
df['Price_tier'].value_counts(normalize=True).round(3)

<!-- The numbers in this cell come from the previous run; re-check them after re-running the notebook. -->
The classes are reasonably balanced: Cheap about 45%, Affordable about 29% and Expensive about 26%. A model that always predicts “Cheap” would already reach about 45% accuracy, so this is the baseline the models need to beat.

In [ ]:
tier_fig = px.histogram(df, x='Price_USD', color='Price_tier', nbins=40, marginal='box',
                        category_orders={'Price_tier': ['Cheap', 'Affordable', 'Expensive']},
                        title='Price Distribution by Tier')
tier_fig.update_layout(bargap=0.05, width=900, height=600, title_x=0.5)
tier_fig.show()

## 4.2 Features from the text columns

In [ ]:
df.select_dtypes(exclude='number').nunique()

In [ ]:
df[['ScreenResolution', 'Cpu', 'Memory', 'Gpu']].head(10)

ScreenResolution, Cpu, Memory and Gpu are free text with many unique values (up to 118). Encoding them directly would create hundreds of dummy columns for only about 1,300 laptops, so the useful information is extracted into a few compact features instead.

In [ ]:
# Screen
df['Touchscreen'] = df['ScreenResolution'].str.contains('Touchscreen').astype(int)
df['IPS'] = df['ScreenResolution'].str.contains('IPS').astype(int)
res = df['ScreenResolution'].str.extract(r'(\d+)x(\d+)').astype(int)
df['X_res'] = res[0]
df['Y_res'] = res[1]

# CPU
def cpu_brand(text):
    if 'Core i7' in text: return 'Core i7'
    if 'Core i5' in text: return 'Core i5'
    if 'Core i3' in text: return 'Core i3'
    if 'AMD' in text: return 'AMD'
    if any(k in text for k in ['Celeron', 'Pentium', 'Atom', 'Samsung']): return 'Low-end'
    return 'Other Intel'      # Core M and Xeon

df['Cpu_brand'] = df['Cpu'].apply(cpu_brand)
df['Cpu_GHz'] = df['Cpu'].str.extract(r'(?i)(\d+\.?\d*)\s*GHz', expand=False).astype(float)

# GPU (the single ARM GPU is grouped with Intel integrated graphics)
df['Gpu_brand'] = df['Gpu'].str.split().str[0].replace('ARM', 'Intel')

# Storage
df['SSD'] = df['Memory'].str.contains('SSD').astype(int)
df['HDD'] = df['Memory'].str.contains('HDD').astype(int)

In [ ]:
print(df['Cpu_brand'].value_counts(), '\n')
print(df['Gpu_brand'].value_counts(), '\n')
print('Missing Cpu_GHz values:', df['Cpu_GHz'].isnull().sum())

In [ ]:
df.groupby('Cpu_brand')['Price_USD'].agg(['mean', 'count']).round(2).sort_values('mean')

<!-- The numbers in this cell come from the previous run; re-check them after re-running the notebook. -->
Average price rises with processor tier: Low-end processors (Celeron, Pentium, Atom) average about 195 USD, Core i3 about 303 USD, Core i5 about 568 USD and Core i7 about 893 USD. AMD laptops (about 314 USD) are comparable to Core i3, while the small “Other Intel” group (Core M and Xeon, 23 laptops) is the most expensive at about 961 USD. The original “Other Intel” category mixed very cheap and very expensive processors, so it was split into two groups. One laptop with a Samsung (ARM) processor was grouped with Low-end, since a single record is too small to form its own category. Part of these price differences likely comes from other specifications that accompany higher-end processors, such as RAM and laptop type.

The original text columns (ScreenResolution, Cpu, Memory and Gpu) are dropped because their useful information has been extracted into new features, and because they contain many unique values that would create too many dummy columns.

In [ ]:
df = df.drop(columns=['ScreenResolution', 'Cpu', 'Memory', 'Gpu'], errors='ignore')
df.info()

# 5. Modeling

## 5.1 Prepare the data

Price_USD is excluded from the features, because the target is defined from it and keeping it would leak the answer to the model. The categorical columns are one-hot encoded (dropping the first category of each to avoid redundant columns). The data is split 80/20, stratified so each tier has the same proportion in the training and test sets.

In [ ]:
X = df.drop(columns=['Price_tier', 'Price_USD'])
y = df['Price_tier']

X = pd.get_dummies(X, columns=['Company', 'TypeName', 'OpSys', 'Cpu_brand', 'Gpu_brand'],
                   drop_first=True, dtype=int)

print(X.shape)
X.dtypes.value_counts()

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

## 5.2 Train and compare models

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

models = {
    'Logistic Regression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)),
    'Random Forest': RandomForestClassifier(max_depth=8, min_samples_leaf=5, random_state=42),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    cv = cross_val_score(model, X_train, y_train, cv=5)
    print(name,
          '| train:', round(model.score(X_train, y_train), 3),
          '| test:', round(model.score(X_test, y_test), 3),
          '| CV:', round(cv.mean(), 3))

<!-- The numbers in this cell come from the previous run; re-check them after re-running the notebook. -->
Both models perform well above the 45% baseline of always predicting “Cheap”. Features are scaled for Logistic Regression, and the Random Forest depth is limited: in an initial run without limits it memorized the training data (train accuracy 0.99 vs test 0.77). After the changes the gap between train and test scores is much smaller (Random Forest: train 0.84 vs test 0.77). Both models reach about 79% cross-validation accuracy, and Random Forest is only slightly better (CV 0.790 vs 0.787), so the difference is too small to call a clear winner. Random Forest is used for the rest of the analysis because it also provides feature importances.

## 5.3 Evaluation

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

pred = models['Random Forest'].predict(X_test)
print(classification_report(y_test, pred))

tiers = ['Cheap', 'Affordable', 'Expensive']
cm = pd.DataFrame(confusion_matrix(y_test, pred, labels=tiers),
                  index=[f'Actual {t}' for t in tiers],
                  columns=[f'Predicted {t}' for t in tiers])
cm

<!-- The numbers in this cell come from the previous run; re-check them after re-running the notebook. -->
The Random Forest reaches 77% accuracy on the test set. Cheap laptops are predicted best (F1 0.85), while the Affordable tier is the hardest (F1 0.62), since it is a narrow price band between two other tiers. Almost all errors (60 of 61) occur between neighbouring tiers: only one Cheap laptop was predicted as Expensive and no Expensive laptop was predicted as Cheap. The model's mistakes are therefore concentrated near the price boundaries (500 and 800 USD) rather than being random.

## 5.4 Feature importance

In [ ]:
imp = pd.Series(models['Random Forest'].feature_importances_, index=X.columns)
imp.sort_values(ascending=False).head(10)

In [ ]:
top = imp.sort_values(ascending=False).head(10).sort_values()
fig_ml = px.bar(top, orientation='h', title='Top 10 Feature Importances (Random Forest)',
                labels={'value': 'Importance', 'index': 'Feature'},
                color_discrete_sequence=['#2c3e50'])
fig_ml.update_layout(width=800, height=600, title_x=0.5, showlegend=False)
fig_ml.show()

<!-- The numbers in this cell come from the previous run; re-check them after re-running the notebook. -->
Ram is by far the most important feature (0.23), followed by whether the laptop is a Notebook (0.12), SSD storage (0.09) and weight (0.08). Processor features (Core i7, clock speed) and screen features (resolution, size) also contribute. This agrees with the EDA, where Ram had the strongest correlation with price. The features extracted from the Cpu, Memory and ScreenResolution columns appear among the top 10, showing that the feature engineering added useful information. Feature importance shows which features the model relies on, not the direction of the effect or causation.

# 6. Conclusion

**Summary**
- The data was cleaned (RAM and weight converted to numbers, prices converted to USD) and rare categories were grouped to avoid unreliable estimates.
- Useful features were extracted from the screen, CPU, GPU and storage text columns, and prices were grouped into three tiers.
- A Random Forest classifier reaches about 77% test accuracy (about 79% in cross-validation), compared with a 45% baseline.
- Almost all errors are between neighbouring tiers, and the Affordable tier is the hardest to predict.
- RAM, laptop type, SSD storage, weight and processor are the most important features.

**Limitations**
- Prices are converted with an approximate exchange rate, and the data is probably several years old, so the tier thresholds are approximate.
- The tier boundaries are a judgment call, so laptops close to a boundary are easily misclassified.
- The dataset is small (about 1,300 laptops), and the models were not tuned beyond limiting the tree depth.

**Possible next steps**
- Predict the exact price with a regression model and compare it with the tier-based approach.
- Tune hyperparameters and try gradient boosting.
- Check feature importance with permutation importance, which is less biased towards continuous features.